<a href="https://colab.research.google.com/github/paulaprado1904/AlgoritmoEvolutivo/blob/main/AEMMT_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math
import time
import numpy as np
import pandas as pd
import os
from datetime import datetime
import matplotlib.pyplot as plt

# Contador global de avaliações completas da função objetivo.
# Esse valor é usado como critério de parada e como métrica de custo computacional.
GLOBAL_AVALIACOES = 0


# ======================================================
# === Estrutura do indivíduo (representação absoluta) ===
# ======================================================
class Individuo:
    """
    Representa uma solução candidata para o problema de dobramento proteico
    no modelo HP 3D.

    A solução é codificada por uma sequência de movimentos absolutos
    em uma malha cúbica tridimensional. A partir desses movimentos,
    o algoritmo reconstrói as coordenadas da cadeia e calcula as métricas
    usadas na avaliação.
    """
    def __init__(self, movimentos):
        # Cromossomo do indivíduo: sequência de movimentos absolutos
        self.movimentos = movimentos[:]

        # Coordenadas reconstruídas da cadeia
        self.coords = None

        # Número de colisões (sobreposição de resíduos na malha)
        self.nC = 0

        # Número de contatos H-H não consecutivos
        self.nH = 0

        # Valor da função objetivo principal
        self.f = None

        # Métricas de compactação
        self.dmax = None
        self.davg = None

        # Métrica combinada usada em uma das tabelas do AEMMT
        self.delta = None

        # Energia simplificada baseada em distâncias entre resíduos H
        self.energiaSimplif = None

        # Energia usada no acompanhamento da melhor solução
        self.energia = None

    def __str__(self):
        """
        Retorna uma representação textual resumida do indivíduo.
        Útil para inspeção manual e depuração.
        """
        f_val = self.f if self.f is not None else 0
        d_max = self.dmax if self.dmax is not None else 0
        d_avg = self.davg if self.davg is not None else 0
        delta_val = self.delta if self.delta is not None else 0
        energia_val = self.energia if self.energia is not None else 0
        energia_simplif_val = self.energiaSimplif if self.energiaSimplif is not None else 0

        return (
            f"f(c): {f_val:3} | "
            f"nC: {self.nC:2} | "
            f"dmax: {d_max:5.2f} | "
            f"davg: {d_avg:5.2f} | "
            f"delta: {delta_val:6.2f} | "
            f"Energia: {energia_val} | "
            f"energiaSimplif: {energia_simplif_val:6.2f} | "
            f"Movimentos: {self.movimentos}"
        )


# ======================================================
# === Conversão de movimento absoluto para deslocamento ===
# ======================================================
def delta_from_move_3d(mov):
    """
    Converte o código inteiro de um movimento em seu deslocamento
    cartesiano correspondente na malha 3D.

    Convenção adotada:
    0 -> +x
    1 -> +y
    2 -> +z
    3 -> -z
    4 -> -y
    5 -> -x
    """
    if mov == 0:   # U: +x
        return (1, 0, 0)
    if mov == 1:   # L: +y
        return (0, 1, 0)
    if mov == 2:   # F: +z
        return (0, 0, 1)
    if mov == 3:   # B: -z
        return (0, 0, -1)
    if mov == 4:   # R: -y
        return (0, -1, 0)
    if mov == 5:   # D: -x
        return (-1, 0, 0)
    raise ValueError("mov inválido (esperado 0..5)")


# ======================================================
# === Construção da conformação e contagem de colisões ===
# ======================================================
def construir_coordenadas(ind):
    """
    Reconstrói as coordenadas da proteína a partir da sequência
    de movimentos do indivíduo.

    Retorna:
    - coords: lista com as posições ocupadas pela cadeia
    - nC: número de colisões detectadas

    Observação:
    uma colisão ocorre quando dois ou mais resíduos ocupam
    a mesma posição na malha.
    """
    x = y = z = 0
    coords = [(0, 0, 0)]
    ocupacao = {(0, 0, 0): 1}

    for mov in ind.movimentos:
        dx, dy, dz = delta_from_move_3d(mov)
        x += dx
        y += dy
        z += dz
        pt = (x, y, z)
        coords.append(pt)
        ocupacao[pt] = ocupacao.get(pt, 0) + 1

    nC = sum(1 for v in ocupacao.values() if v > 1)

    ind.coords = coords
    ind.nC = nC
    return coords, nC


def avaliar_base(ind, cadeia, rho=1.0):
    """
    Avaliação completa do indivíduo.

    Métricas calculadas:
    - nC: número de colisões
    - nH: número de contatos H-H não consecutivos
    - f(c): função objetivo principal
    - dmax: compactação baseada na maior distância entre pares H-H
    - davg: compactação baseada na distância média entre pares H-H
    - energiaSimplif: soma das distâncias entre resíduos H
    - energia: valor auxiliar usado para acompanhar a melhor solução

    Regras:
    - Soluções com colisão são consideradas inviáveis.
    - Ainda assim, soluções inviáveis recebem valores de compactação penalizados,
      para que a informação estrutural não seja totalmente descartada.
    """
    global GLOBAL_AVALIACOES
    GLOBAL_AVALIACOES += 1

    PENAL = 1e6

    coords, nC = construir_coordenadas(ind)
    n = len(cadeia)

    # Salva as coordenadas reconstruídas no indivíduo
    ind.coords = coords

    # Caso inconsistente: conformação inválida
    if coords is None or len(coords) != n:
        ind.nC = max(1, int(nC) if nC is not None else 1)
        ind.nH = 0
        ind.f = -rho * ind.nC
        ind.dmax = float('inf')
        ind.davg = float('inf')
        ind.delta = None
        ind.energia = -ind.f
        return

    # --------------------------------------------------
    # Cálculo das distâncias entre pares de resíduos H
    # não consecutivos na cadeia
    # --------------------------------------------------
    H_idx = [i for i, a in enumerate(cadeia) if a == "H"]
    dists = []
    m = len(H_idx)
    Es = 0.0

    for a in range(m):
        i = H_idx[a]
        for b in range(a + 1, m):
            j = H_idx[b]

            # Ignora resíduos H adjacentes na cadeia
            if abs(i - j) == 1:
                continue

            p1 = coords[i]
            p2 = coords[j]

            # Ignora pares que ocupam a mesma posição
            if p1 == p2:
                continue

            d = math.sqrt(
                (p1[0] - p2[0]) ** 2 +
                (p1[1] - p2[1]) ** 2 +
                (p1[2] - p2[2]) ** 2
            )

            dists.append(d)
            Es += d

    if not dists:
        dmax = float('inf')
        davg = float('inf')
    else:
        dmax = max(dists)
        davg = sum(dists) / len(dists)

    # --------------------------------------------------
    # Caso inviável: há colisões
    # --------------------------------------------------
    if nC > 0:
        ind.nC = nC
        ind.nH = 0
        ind.f = -rho * nC

        # Mesmo em indivíduos inviáveis, mantém-se uma noção de compactação
        # incorporando a penalização por colisão
        ind.dmax = 1 / (dmax + rho * nC)
        ind.davg = 1 / (davg + rho * nC)

        ind.delta = None
        Es = Es + (PENAL * nC)
        ind.energiaSimplif = Es
        ind.energia = -ind.f
        return

    # --------------------------------------------------
    # Caso viável: sem colisões
    # Conta contatos H-H não consecutivos e adjacentes no espaço
    # --------------------------------------------------
    nH = 0
    for i in range(n):
        if cadeia[i] != "H":
            continue
        for j in range(i + 2, n):
            if cadeia[j] != "H":
                continue

            pi = coords[i]
            pj = coords[j]
            manh = abs(pi[0] - pj[0]) + abs(pi[1] - pj[1]) + abs(pi[2] - pj[2])

            if manh == 1:
                nH += 1

    ind.nC = 0
    ind.nH = nH
    ind.f = nH
    ind.dmax = 1 / dmax
    ind.davg = 1 / davg
    ind.delta = None
    ind.energiaSimplif = Es
    ind.energia = -ind.f


# ======================================================
# === Atualização da métrica delta ===
# ======================================================
def atualizar_delta(populacao, prev_stats):
    """
    Atualiza a métrica delta dos indivíduos da população.

    O algoritmo calcula as médias populacionais de dmax e davg
    e compara sua variação em relação à geração anterior.
    A métrica que mais variou passa a ser a métrica ativa.

    Em seguida, para cada indivíduo:
        delta(c) = f(c) * d(c)

    em que d(c) pode ser dmax ou davg, dependendo da métrica ativa.
    """
    # Médias da geração atual
    dmax_vals = np.array([ind.dmax for ind in populacao], dtype=float)
    davg_vals = np.array([ind.davg for ind in populacao], dtype=float)

    dmax_fin = dmax_vals[np.isfinite(dmax_vals)]
    davg_fin = davg_vals[np.isfinite(davg_vals)]

    mean_dmax = float(np.mean(dmax_fin)) if dmax_fin.size else float("inf")
    mean_davg = float(np.mean(davg_fin)) if davg_fin.size else float("inf")

    # Informações da geração anterior
    prev_dmax = float(prev_stats.get("mean_dmax", float("inf"))) if prev_stats else float("inf")
    prev_davg = float(prev_stats.get("mean_davg", float("inf"))) if prev_stats else float("inf")
    active_metric_prev = prev_stats.get("active_metric", "dmax") if prev_stats else "dmax"

    def razao(atual, anterior):
        """
        Mede a variação relativa entre duas gerações.
        Retorna |atual/anterior - 1|.
        """
        if (not np.isfinite(atual)) or (not np.isfinite(anterior)) or anterior <= 0:
            return None
        return abs(atual / anterior - 1)

    r_dmax = razao(mean_dmax, prev_dmax)
    r_davg = razao(mean_davg, prev_davg)

    # Escolhe a métrica ativa da geração atual
    active_metric = escolhe_metrica_por_variacao(r_dmax, r_davg, atual=active_metric_prev)

    # Atualiza delta de cada indivíduo com base na métrica ativa
    for ind in populacao:
        if ind.f is None or not np.isfinite(ind.f):
            ind.delta = None
            continue

        d_ind = ind.dmax if active_metric == "dmax" else ind.davg
        if (d_ind is None) or (not np.isfinite(d_ind)) or d_ind <= 0:
            ind.delta = None
            continue

        ind.delta = ind.f * d_ind

        # Armazena a métrica usada apenas para depuração/análise
        ind.d = d_ind

    return {
        "mean_dmax": mean_dmax,
        "mean_davg": mean_davg,
        "active_metric": active_metric,
        "r_dmax": r_dmax,
        "r_davg": r_davg,
    }


def escolhe_metrica_por_variacao(r_dmax, r_davg, atual="dmax"):
    """
    Escolhe a métrica de compactação a ser usada no delta.

    Critério:
    - Se apenas uma métrica for válida, escolhe essa.
    - Se ambas forem válidas, escolhe a que mais variou.
    - Em caso de empate, mantém a métrica anterior.
    """
    if r_dmax is None and r_davg is None:
        return atual
    if r_dmax is None:
        return "davg"
    if r_davg is None:
        return "dmax"

    var_dmax = abs(r_dmax - 1.0)
    var_davg = abs(r_davg - 1.0)

    if abs(var_dmax - var_davg) < 1e-15:
        return atual

    return "dmax" if var_dmax > var_davg else "davg"


# ======================================================
# === Geração aleatória de indivíduo ===
# ======================================================
def gerar_individuo_aleatorio(n_residuos):
    """
    Gera um indivíduo aleatório com n_residuos - 1 movimentos.

    O movimento 3 (-z) é excluído da geração inicial para reduzir
    simetrias redundantes na representação das conformações.
    """
    L = n_residuos - 1
    moves = [0, 1, 2, 4, 5]
    movimentos = [random.choice(moves) for _ in range(L)]
    return Individuo(movimentos)


# ======================================================
# === Recombinação por múltiplos pontos ===
# ======================================================
def crossover_multipontos(pai, mae, n_residuos):
    """
    Gera um único filho a partir de crossover de múltiplos pontos
    entre dois pais.
    """
    L = len(pai.movimentos)
    if L != len(mae.movimentos):
        raise ValueError("Pais com comprimentos diferentes de cromossomo")

    num_cortes = int(round(n_residuos / 10.0))
    if num_cortes < 1:
        num_cortes = 1
    if num_cortes > L - 1:
        num_cortes = L - 1

    cortes = sorted(random.sample(range(1, L), num_cortes))

    filho_mov = []
    pais = [pai.movimentos, mae.movimentos]
    idx_pai = 0
    inicio = 0

    for c in cortes + [L]:
        segmento = pais[idx_pai][inicio:c]
        filho_mov.extend(segmento)
        idx_pai = 1 - idx_pai
        inicio = c

    return Individuo(filho_mov)


def crossover_multipontos_2filhos(pai, mae, n_residuos):
    """
    Gera dois filhos por crossover de múltiplos pontos.
    Cada filho começa copiando segmentos de um pai diferente.
    """
    L = len(pai.movimentos)
    if L != len(mae.movimentos):
        raise ValueError("Pais com comprimentos diferentes de cromossomo")

    num_cortes = int(round(n_residuos / 10.0))
    num_cortes = max(1, min(num_cortes, L - 1))
    cortes = sorted(random.sample(range(1, L), num_cortes))

    def build_child(start_with_pai: bool):
        filho_mov = []
        pais = [pai.movimentos, mae.movimentos]
        idx = 0 if start_with_pai else 1
        ini = 0
        for c in cortes + [L]:
            filho_mov.extend(pais[idx][ini:c])
            idx = 1 - idx
            ini = c
        return Individuo(filho_mov)

    return build_child(True), build_child(False)


# ======================================================
# === Mutação gene a gene ===
# ======================================================
def mutacao_por_gene(ind, taxa_mut):
    """
    Aplica mutação gene a gene.

    Cada posição do cromossomo é alterada com probabilidade taxa_mut,
    sendo substituída por um novo movimento aleatório.
    """
    novo = Individuo(ind.movimentos[:])
    for j in range(len(novo.movimentos)):
        if random.random() <= taxa_mut:
            novo.movimentos[j] = random.randint(0, 5)
    return novo


# ======================================================
# === Seleção de pais ===
# ======================================================
def selecao_pai_pool_unico(tabelas, k=3):
    """
    Une os indivíduos presentes em todas as tabelas e realiza
    seleção por torneio com base no atributo delta.
    """
    pool = unir_tabelas(*[t for t in tabelas if t])
    if not pool:
        raise RuntimeError("Pool vazio: todas as tabelas estão vazias.")

    # Considera apenas indivíduos com delta já calculado
    pool_validos = [ind for ind in pool if getattr(ind, "delta", None) is not None]

    if len(pool_validos) >= k:
        return selecao_torneio(pool_validos, k=k, atributo="delta", maximize=True)

    if pool_validos:
        return selecao_torneio(pool_validos, k=len(pool_validos), atributo="delta", maximize=True)

    # Se ninguém ainda possui delta válido, escolhe aleatoriamente
    return random.choice(pool)


def selecao_pai_pool_unico2(tabelas, k=3):
    """
    Une os indivíduos presentes nas tabelas e escolhe um pai
    aleatoriamente no pool resultante.
    """
    pool = unir_tabelas(*[t for t in tabelas if t])
    if not pool:
        raise RuntimeError("Pool vazio: todas as tabelas estão vazias.")
    return random.choice(pool)


def selecao_torneio(pop, k=3, atributo="delta", maximize=True):
    """
    Seleção por torneio baseada em um atributo do indivíduo.
    """
    if not pop:
        raise RuntimeError("População vazia no torneio.")

    cand = random.sample(pop, k=min(k, len(pop)))

    # Remove candidatos sem valor válido para o atributo usado no torneio
    cand = [ind for ind in cand if getattr(ind, atributo, None) is not None]
    if not cand:
        return random.choice(pop)

    key_fn = lambda ind: getattr(ind, atributo)
    return max(cand, key=key_fn) if maximize else min(cand, key=key_fn)


def assinatura_ind(ind: Individuo):
    """
    Gera uma assinatura imutável do indivíduo com base no cromossomo.
    É usada para evitar genótipos duplicados.
    """
    return tuple(ind.movimentos)


# ======================================================
# === Inserção nas tabelas do AEMMT ===
# ======================================================
def inserir_tabela(tabela, ind, tamanho, maximize, atributo):
    """
    Insere um indivíduo em uma tabela especializada, respeitando:
    - unicidade por cromossomo;
    - limite máximo de tamanho;
    - critério de maximização ou minimização do atributo.
    """
    sig = assinatura_ind(ind)
    if any(assinatura_ind(x) == sig for x in tabela):
        return

    val = getattr(ind, atributo, None)
    if val is None:
        return

    if len(tabela) < tamanho:
        tabela.append(ind)
        return

    if maximize:
        pior = min(tabela, key=lambda x: getattr(x, atributo))
        melhor_no_attr = val > getattr(pior, atributo)
    else:
        pior = max(tabela, key=lambda x: getattr(x, atributo))
        melhor_no_attr = val < getattr(pior, atributo)

    if not melhor_no_attr:
        return

    tabela.remove(pior)
    tabela.append(ind)

def valor_objetivo(ind, atributo):
    """
    Retorna o valor de um objetivo do indivíduo.
    Se não houver valor válido, retorna -inf para maximização.
    """
    v = getattr(ind, atributo, None)
    if v is None:
        return -float("inf")
    if isinstance(v, (int, float)) and np.isfinite(v):
        return float(v)
    return -float("inf")


def domina(a, b, objetivos=("f", "dmax", "davg", "delta")):
    """
    Verifica se o indivíduo 'a' domina o indivíduo 'b'
    considerando todos os objetivos em maximização.
    """
    vals_a = [valor_objetivo(a, obj) for obj in objetivos]
    vals_b = [valor_objetivo(b, obj) for obj in objetivos]

    melhor_ou_igual_em_todos = all(x >= y for x, y in zip(vals_a, vals_b))
    melhor_em_pelo_menos_um = any(x > y for x, y in zip(vals_a, vals_b))

    return melhor_ou_igual_em_todos and melhor_em_pelo_menos_um

def inserir_tabela_nd(tabela_nd, ind, tamanho_nd, objetivos=("f", "dmax", "davg", "delta")):
    """
    Insere incrementalmente um indivíduo na tabela de não-dominância.

    Regras:
    - não insere duplicado por cromossomo;
    - se for dominado por alguém da tabela, não entra;
    - se dominar indivíduos da tabela, remove os dominados;
    - se for não dominado e não dominar ninguém, entra;
    - se exceder o tamanho, aplica desempate.
    """
    sig = assinatura_ind(ind)

    if any(assinatura_ind(x) == sig for x in tabela_nd):
        return

    # Se algum da tabela domina o novo, ele não entra
    for atual in tabela_nd:
        if domina(atual, ind, objetivos):
            return

    # Remove da tabela todos os que forem dominados pelo novo
    dominados = [atual for atual in tabela_nd if domina(ind, atual, objetivos)]

    for d in dominados:
        tabela_nd.remove(d)

    # Entra na tabela
    tabela_nd.append(ind)

    # Se ultrapassar o tamanho máximo, desempata
    if len(tabela_nd) > tamanho_nd:
        def score_nd(x):
            return sum(valor_objetivo(x, obj) for obj in objetivos)

        pior = min(tabela_nd, key=score_nd)
        tabela_nd.remove(pior)


def avaliar_parcial_nH_nC(ind, cadeia):
    """
    Avaliação parcial e barata usada antes da avaliação completa.

    Calcula apenas:
    - nC: número de colisões
    - nH: número de contatos H-H, apenas se não houver colisão

    Não incrementa o contador GLOBAL_AVALIACOES.
    Retorna (nC, nH).
    """
    coords, nC = construir_coordenadas(ind)
    n = len(cadeia)

    if coords is None or len(coords) != n:
        return 1, 0

    if nC > 0:
        return nC, 0

    nH = 0
    for i in range(n):
        if cadeia[i] != "H":
            continue
        for j in range(i + 2, n):
            if cadeia[j] != "H":
                continue
            pi = coords[i]
            pj = coords[j]
            manh = abs(pi[0] - pj[0]) + abs(pi[1] - pj[1]) + abs(pi[2] - pj[2])
            if manh == 1:
                nH += 1

    return 0, nH


def unir_tabelas(*tabelas):
    """
    Une várias tabelas em uma única lista, sem repetir objetos.
    """
    uniao = []
    seen = set()
    for tab in tabelas:
        for ind in tab:
            if id(ind) not in seen:
                seen.add(id(ind))
                uniao.append(ind)
    return uniao


def salvar_grafico_melhor_energia_com_marcador(df_hist, pasta_saida, nome_proteina, execucao_id):
    """
    Salva um gráfico da evolução da melhor energia ao longo das gerações,
    destacando a geração em que a melhor solução foi encontrada.
    """
    if df_hist is None or df_hist.empty:
        print("[WARN] df_hist vazio.")
        return None
    if "Iteracao" not in df_hist.columns or "Melhor Energia" not in df_hist.columns:
        print("[WARN] df_hist sem colunas necessárias (Iteracao, Melhor Energia).")
        return None

    xs = df_hist["Iteracao"].to_numpy()
    ys = df_hist["Melhor Energia"].to_numpy()

    idx_best = int(df_hist["Melhor Energia"].idxmin())
    best_gen = int(df_hist.loc[idx_best, "Iteracao"])
    best_E = float(df_hist.loc[idx_best, "Melhor Energia"])

    plt.figure(figsize=(8, 4.5))
    plt.plot(xs, ys)
    plt.axvline(best_gen, linestyle="--", linewidth=1)
    plt.scatter([best_gen], [best_E], s=60)

    plt.xlabel("Geração")
    plt.ylabel("Melhor energia")
    plt.title(f"{nome_proteina} — exec {execucao_id} — melhor gen {best_gen} (E={best_E:.0f})")
    plt.grid(True, alpha=0.3)

    ultima_gen = int(df_hist["Iteracao"].max())
    nome_arquivo = f"{nome_proteina}_exec{execucao_id:02d}_ate{ultima_gen:04d}_bestgen{best_gen:04d}_energia.png"
    caminho_fig = os.path.join(pasta_saida, nome_arquivo)

    plt.tight_layout()
    plt.savefig(caminho_fig, dpi=200)
    plt.close()
    return caminho_fig


# ======================================================
# === Algoritmo AEMMT 4 para HP 3D sem Monte Carlo ===
# ======================================================
def aemt_hp_3d_sem_mc(
    cadeia_raw,
    pop_init=10,
    taxa_mut=0.2,
    rho=1.0,
    subpop_sizes=(1, 2, 2, 3, 2),
    torneio_k=2,
    taxa_cross=1.0,
    energia_otima=None,
    max_avaliacoes=None,
    seed=None
):
    """
    Executa o algoritmo evolutivo multiobjetivo com 4 tabelas (AEMMT 4).

    Estrutura geral:
    1. Gera população inicial
    2. Avalia os indivíduos
    3. Atualiza o delta
    4. Distribui os indivíduos em 4 tabelas especializadas:
       - T_f
       - T_dmax
       - T_davg
       - T_delta
    5. Gera descendentes por seleção, crossover e mutação
    6. Atualiza as tabelas
    7. Registra estatísticas até atingir o critério de parada
    """
    global GLOBAL_AVALIACOES

    cadeia = ''.join([c for c in cadeia_raw.strip().upper() if c in ('H', 'P')])
    n_res = len(cadeia)
    if n_res < 2:
        raise ValueError("Cadeia muito curta")

    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # ---------------- População inicial ----------------
    populacao = []
    for _ in range(pop_init):
        if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
            break
        ind = gerar_individuo_aleatorio(n_res)
        avaliar_base(ind, cadeia, rho=rho)
        populacao.append(ind)

    if len(populacao) == 0:
        raise RuntimeError("Nenhum indivíduo pôde ser avaliado (max_avaliacoes muito baixo).")

    prev_stats = atualizar_delta(populacao, prev_stats=None)

    size_f, size_dmax, size_davg, size_delta, size_nd = subpop_sizes
    T_f, T_dmax, T_davg, T_delta, T_nd = [], [], [], [], []

    for ind in populacao:
        inserir_tabela(T_f, ind, size_f, True, 'f')
        inserir_tabela(T_dmax, ind, size_dmax, True, 'dmax')
        inserir_tabela(T_davg, ind, size_davg, True, 'davg')
        inserir_tabela(T_delta, ind, size_delta, True, 'delta')
        inserir_tabela_nd(T_nd, ind, size_nd, objetivos=("f", "dmax", "davg", "delta"))

    # Pool inicial: união das 4 tabelas
    pool0 = unir_tabelas(T_f, T_dmax, T_davg, T_delta, T_nd)

    # Melhor indivíduo global até o momento
    melhor_global = min(pool0, key=lambda ind: ind.energia)
    geracao_melhor = 0
    hit_otimo = False
    aval_ate_otimo = None

    if energia_otima is not None and melhor_global.energia <= energia_otima:
        hit_otimo = True
        aval_ate_otimo = GLOBAL_AVALIACOES

    # Histórico por geração
    melhores_energia = []
    medias_energia = []
    desvios_energia = []

    energias0 = [ind.energia for ind in pool0]
    medias_energia.append(float(np.mean(energias0)))
    desvios_energia.append(float(np.std(energias0)))
    melhores_energia.append(float(melhor_global.energia))

    # ------------- Loop principal -------------
    geracao = 0
    while True:
        if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
            break
        if energia_otima is not None and hit_otimo:
            break

        tabelas = [T_f, T_dmax, T_davg, T_delta, T_nd]

        nova_pop = []
        assinaturas = set()

        while len(nova_pop) < pop_init:
            if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
                break
            if energia_otima is not None and hit_otimo:
                break

            pai = selecao_pai_pool_unico(tabelas, 1)
            mae = selecao_pai_pool_unico2(tabelas, 1)

            # Recombinação
            if random.random() < taxa_cross:
                base_f1, base_f2 = crossover_multipontos_2filhos(pai, mae, n_res)
            else:
                base_f1, base_f2 = Individuo(pai.movimentos[:]), Individuo(mae.movimentos[:])

            for base in (base_f1, base_f2):
                if len(nova_pop) >= pop_init:
                    break

                # Filho base sem mutação
                cand0 = Individuo(base.movimentos[:])

                # Avaliação parcial barata para decidir se muta
                nC0, nH0 = avaliar_parcial_nH_nC(cand0, cadeia)

                # Referência: nH do melhor indivíduo atual
                best_nH = melhor_global.nH if getattr(melhor_global, "nH", None) is not None else 0

                # Regra: só aceita sem mutação se superar o melhor nH atual
                precisa_mutar = not (nH0 > best_nH)

                filho_final = None

                # Se não precisar mutar, tenta aceitar diretamente
                if not precisa_mutar:
                    sig0 = assinatura_ind(cand0)
                    if sig0 not in assinaturas:
                        filho_final = cand0
                        assinaturas.add(sig0)
                    else:
                        # Se repetiu, força mutação para manter diversidade
                        precisa_mutar = True

                # Se precisar mutar, tenta gerar uma assinatura nova
                if precisa_mutar:
                    for _tent in range(10):
                        cand = mutacao_por_gene(cand0, taxa_mut)
                        sig = assinatura_ind(cand)
                        if sig not in assinaturas:
                            filho_final = cand
                            assinaturas.add(sig)
                            break

                    # Fallback: aceita mesmo sem garantir total diversidade
                    if filho_final is None:
                        filho_final = mutacao_por_gene(cand0, taxa_mut)
                        sig = assinatura_ind(filho_final)
                        assinaturas.add(sig)

                # Avaliação completa do descendente
                avaliar_base(filho_final, cadeia, rho=rho)
                nova_pop.append(filho_final)

                # Verifica se atingiu o ótimo conhecido
                if energia_otima is not None and not hit_otimo:
                    if filho_final.energia <= energia_otima:
                        hit_otimo = True
                        aval_ate_otimo = GLOBAL_AVALIACOES
                        melhor_global = filho_final
                        geracao_melhor = geracao + 1

        if len(nova_pop) == 0:
            break

        prev_stats = atualizar_delta(nova_pop, prev_stats)

        for ind in nova_pop:
            inserir_tabela(T_f, ind, size_f, True, 'f')
            inserir_tabela(T_dmax, ind, size_dmax, True, 'dmax')
            inserir_tabela(T_davg, ind, size_davg, True, 'davg')
            inserir_tabela(T_delta, ind, size_delta, True, 'delta')
            inserir_tabela_nd(T_nd, ind, size_nd, objetivos=("f", "dmax", "davg", "delta"))

        # Pool acumulado: união das tabelas após atualização
        pool = unir_tabelas(T_f, T_dmax, T_davg, T_delta)

        energias = [ind.energia for ind in pool]
        medias_energia.append(float(np.mean(energias)))
        desvios_energia.append(float(np.std(energias)))

        melhor_ger = min(pool, key=lambda ind: ind.energia)
        melhores_energia.append(float(melhor_ger.energia))

        if melhor_ger.energia < melhor_global.energia:
            melhor_global = melhor_ger
            geracao_melhor = geracao + 1
            if energia_otima is not None and melhor_global.energia <= energia_otima and not hit_otimo:
                hit_otimo = True
                aval_ate_otimo = GLOBAL_AVALIACOES

        populacao = nova_pop
        geracao += 1

    resumo = {
        "Menor Energia Obtida": melhor_global.energia,
        "Melhor f(c)": melhor_global.f,
        "nH(c)": melhor_global.nH,
        "nC(c)": melhor_global.nC,
        "dmax(c)": melhor_global.dmax,
        "davg(c)": melhor_global.davg,
        "delta(c)": melhor_global.delta,
        "Geração do Melhor": geracao_melhor,
        "Hit_otimo": hit_otimo,
        "Aval_ate_otimo": aval_ate_otimo,
        "Aval_total": GLOBAL_AVALIACOES,
        "Métrica δ ativa última geração": prev_stats["active_metric"]
    }

    df_hist = pd.DataFrame({
        "Iteracao": list(range(len(melhores_energia))),
        "Energia Média": medias_energia,
        "Desvio Padrão": desvios_energia,
        "Melhor Energia": melhores_energia
    })

    caminho_fig = salvar_grafico_melhor_energia_com_marcador(
        df_hist, pasta_saida, nome_proteina, i + 1
    )
    print(f"[OK] Execução {i+1}: gráfico salvo em {caminho_fig}")

    return df_hist, resumo, (T_f, T_dmax, T_davg, T_delta)


# ======================================================
# === Execução repetida dos experimentos ===
# ======================================================
if __name__ == "__main__":
    # Caminho do arquivo com a cadeia HP e diretório de saída
    caminho_arquivo = "/content/drive/MyDrive/pastaOrigem/273d.10.txt"
    pasta_saida = "/content/drive/MyDrive/pastaDestino/teste/273d10"
    os.makedirs(pasta_saida, exist_ok=True)

    # Leitura do arquivo de entrada:
    # linha 1 -> tamanho da cadeia
    # linha 2 -> sequência HP
    with open(caminho_arquivo) as f:
        n = int(f.readline().strip())
        cadeia = ''.join([c for c in f.readline().strip().upper() if c in ('H', 'P')])
        assert len(cadeia) == n

    nome_proteina = os.path.splitext(os.path.basename(caminho_arquivo))[0]

    # Ótimo conhecido da instância
    ENERGIA_OTIMA = -11

    # Ajuste de parâmetros conforme o tamanho da proteína
    if n <= 36:
        subpop_sizes = (80, 80, 80, 80, 80)
        taxa_mut = 0.1
        taxa_cross = 0.95
        MAX_AVALIACOES = 1_500_000
    else:
        subpop_sizes = (1, 1, 1, 2)
        taxa_mut = 0.40
        taxa_cross = 0.50
        MAX_AVALIACOES = 3_000_000

    NUM_EXECUCOES = 50
    resultados_execucoes = []

    for i in range(NUM_EXECUCOES):
        GLOBAL_AVALIACOES = 0
        random.seed()
        np.random.seed()

        print(f"\n========== EXECUÇÃO {i+1}/{NUM_EXECUCOES} ==========")

        df_hist, resumo, tabelas = aemt_hp_3d_sem_mc(
            cadeia,
            pop_init=400,
            taxa_mut=taxa_mut,
            rho=1.0,
            subpop_sizes=subpop_sizes,
            torneio_k=1,
            taxa_cross=taxa_cross,
            energia_otima=ENERGIA_OTIMA,
            max_avaliacoes=MAX_AVALIACOES
        )

        T_f, T_dmax, T_davg, T_delta = tabelas
        pool_final = unir_tabelas(T_f, T_dmax, T_davg, T_delta)

        # Estatísticas do conjunto final de indivíduos armazenados nas tabelas
        gen_unicos_final = len({tuple(ind.movimentos) for ind in pool_final})
        E_final = np.array([ind.energia for ind in pool_final], dtype=float)
        E_media_final = float(np.mean(E_final))
        E_dp_final = float(np.std(E_final))

        resultados_execucoes.append({
            "Execucao": i + 1,
            "Melhor_Energia": resumo["Menor Energia Obtida"],
            "Hit_otimo": resumo["Hit_otimo"],
            "Aval_ate_otimo": resumo["Aval_ate_otimo"],
            "Aval_total": resumo["Aval_total"],
            "Genotipos_unicos_final": gen_unicos_final,
            "Energia_media_finalpop": E_media_final,
            "Energia_dp_finalpop": E_dp_final,
        })

    df_resumo = pd.DataFrame(resultados_execucoes)

    # Estatísticas agregadas da melhor energia por execução
    energies = df_resumo["Melhor_Energia"].replace([np.inf, -np.inf], np.nan).dropna()

    melhor_energia_media = float(energies.mean()) if not energies.empty else float("nan")
    melhor_energia_dp = float(energies.std(ddof=1)) if len(energies) > 1 else float("nan")

    qtd_facteis = int(energies.shape[0])
    qtd_total = int(df_resumo.shape[0])

    print("\n\n===== RESUMO DAS 50 EXECUÇÕES =====")
    print(df_resumo)

    # Considera apenas execuções que atingiram o ótimo conhecido
    df_sucesso = df_resumo[df_resumo["Hit_otimo"] == True]

    if not df_sucesso.empty:
        media_aval = df_sucesso["Aval_ate_otimo"].mean()
        mediana_aval = df_sucesso["Aval_ate_otimo"].median()
    else:
        media_aval = float('nan')
        mediana_aval = float('nan')

    taxa_acertos = 100.0 * df_sucesso.shape[0] / NUM_EXECUCOES

    print("\n===== RESUMO GLOBAL (estilo artigo) =====")
    print(f"Ótimo conhecido: {ENERGIA_OTIMA}")
    print(f"Menor energia obtida: {df_resumo['Melhor_Energia'].min()}")
    print(f"Acurácia (Ac.): {taxa_acertos:.1f} %")
    print(f"Média Aval. (apenas execuções com ótimo): {media_aval:.0f}")
    print(f"Mediana Aval.: {mediana_aval:.0f}")

    # Salva resumo em arquivo texto
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    caminho_saida_txt = os.path.join(
        pasta_saida,
        f"{nome_proteina}_{timestamp}.txt"
    )

    with open(caminho_saida_txt, "w") as f_out:
        f_out.write(f"Arquivo da proteína: {caminho_arquivo}\n")
        f_out.write(f"Nome da proteína (base): {nome_proteina}\n")
        f_out.write(f"Ótimo conhecido: {ENERGIA_OTIMA}\n\n")

        f_out.write("===== RESUMO DAS 50 EXECUÇÕES =====\n")
        f_out.write(df_resumo.to_string(index=False))
        f_out.write("\n\n===== RESUMO GLOBAL (estilo artigo) =====\n")
        f_out.write(f"Menor energia obtida: {df_resumo['Melhor_Energia'].min()}\n")
        f_out.write(f"Acurácia (Ac.): {taxa_acertos:.1f} %\n")
        f_out.write(f"Média Aval. (execuções com ótimo): {media_aval:.0f}\n")
        f_out.write(f"Mediana Aval.: {mediana_aval:.0f}\n")

        f_out.write("\n----- ESTATÍSTICAS (MELHOR ENERGIA POR EXECUÇÃO) -----\n")
        f_out.write(f"Melhor_Energia (média ± dp): {melhor_energia_media:.4f} ± {melhor_energia_dp:.4f}\n")
        f_out.write(f"Execuções factíveis consideradas: {qtd_facteis}/{qtd_total}\n")

    print(f"\nResumo salvo em: {caminho_saida_txt}")